# Error analysis: profiles, matching and peer review

This notebook reads saved results only. It makes **no LLM calls**, crawls, embeddings or valuation calculations.
It asks where problems may arise: source evidence, facet fills, candidate retrieval, LLM selection or the review rubric.
Human labels are the evaluation reference; model reasons are supporting observations, not ground truth.

**Scope:** pinned B3 run, Haiku/Sonnet profiles and selectors, round-1 labels, completed round-2 labels when available,
one-shot Sol and the two saved agent-loop runs. The old 663-pair 0–3 evaluation remains separate.
Use the uv `.venv` kernel. Derived files are written to a new `outputs-local/v1/error_analysis/<run>/` folder.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_manifest.json").exists())
sys.path.insert(0, str(ROOT / "v1/scripts"))
import error_analysis as ea
importlib.reload(ea)
CONFIG = ea.Config()  # explicit pinned source paths; inspect before changing versions
OUT = None            # None creates a new timestamped folder; no input files are overwritten
analysis = ea.Analysis(CONFIG, OUT).build()
print("Output directory:", analysis.out)
display(Markdown((analysis.out / "report.md").read_text()))


## 1. Which facet claims need checking?

The chart shows **problems in the historical text humans labeled**, not current model accuracy.
A partly correct claim counts as a problem here. “Can't tell” is excluded from the denominator.
Company size is displayed for context but is excluded from B3. The exact-text columns tell us how much
of this benchmark actually applies to current profiles. Different text needs new verification.


In [ ]:
facets = analysis.frames["facet_priorities"].copy()
display(facets)
fig, ax = plt.subplots(figsize=(10, 5))
labels = facets.facet.str.replace("_", " ", regex=False)
ax.barh(labels, facets.historical_problem_rate * 100, color="slategray")
ax.invert_yaxis()
ax.set(title="Problems in human-labeled historical facet claims", xlabel="Problem share of checkable historical claims (%)", ylabel="Facet")
fig.text(.01, .01, "Source: historical facet label sheet; partly correct + wrong + outdated. Not current-fill accuracy.", fontsize=8)
fig.tight_layout(rect=(0, .06, 1, 1))
fig.savefig(analysis.out / "historical_facet_problems.png", dpi=160)
plt.show()
print("Current Haiku exact-text/checkable claims:", facets.current_haiku_exact_checkable.sum())
print("Current Sonnet exact-text/checkable claims:", facets.current_sonnet_exact_checkable.sum())


## 2. Does selection improve or over-filter the peer list?

Use human-majority labels only; unreviewed pairs are not scored as errors. “Known peers retained” measures
retention within this reviewed pool, not recall across all public companies. A selection method can have high
precision by returning almost nothing, so inspect coverage and direct/adjacent peer retention together.
The screen comparisons use the same B3 candidates and Haiku profiles; Sonnet profiles were not used downstream.


In [ ]:
metrics = analysis.frames["method_metrics"]
display(metrics)
if not metrics.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    counts = metrics.pivot(index="method", columns="target_ticker", values="known_good_retained")
    counts.plot.barh(ax=ax, color=["#315c72", "#a2b4be"][:len(counts.columns)] if len(counts.columns) <= 2 else None)
    ax.set(title="Human-labeled direct or adjacent peers retained", xlabel="Number of known peers in returned top 10", ylabel="Method")
    ax.legend(title="Target")
    fig.text(.01, .01, "Source: saved method outputs joined to human-majority labels; known-peer totals are in the table above.", fontsize=8)
    fig.tight_layout(rect=(0, .06, 1, 1))
    fig.savefig(analysis.out / "known_peers_retained.png", dpi=160)
    plt.show()
display(analysis.frames["judge_metrics"])
print("Judge agreement differences do not isolate model, evidence-context or retrieval effects.")


## 3. Review concrete disagreements

Flags describe observed outcomes, not proven causes. For each case, the JSON packet includes both companies'
current Haiku/Sonnet facets, original reviewer scores/notes, historical facet labels and exact-text applicability,
selector reasons, judge reasons and saved website passages. Review agreements as controls as well as errors.
No original profiles are corrected by this notebook.


In [ ]:
queue = analysis.frames["case_queue"]
display(queue)
CASE_ID = None  # e.g. copy one case_id from the queue; None selects the highest-priority case
chosen = queue.loc[queue.case_id.eq(CASE_ID)].iloc[0] if CASE_ID else queue.iloc[0]
case_file = analysis.out / "cases" / f"{chosen.target}__{chosen.comp}.json"
import json
packet = json.loads(case_file.read_text())
print("Case:", case_file)
display(Markdown(f"### {chosen.target_ticker} → {chosen.comp_ticker}\n\nObserved flags: {chosen['flags']}"))
print(json.dumps(packet["references"]["HUMAN"], indent=2, ensure_ascii=False))
for ref in ["METHODS", "JUDGES"]:
    display(pd.DataFrame(packet["references"][ref]["data"]))
profile_rows = []
for key, ref in packet["references"].items():
    if key.startswith(("TARGET:haiku:", "TARGET:sonnet:", "COMP:haiku:", "COMP:sonnet:")):
        side, model, facet = key.split(":")
        data = ref["data"]
        profile_rows.append({"company_side": side, "model": model, "facet": facet,
                             "value": data[facet], "status": data[facet + "_status"],
                             "source": data[facet + "_source"]})
with pd.option_context("display.max_colwidth", 150):
    display(pd.DataFrame(profile_rows))


## 4. Optional manual eval-agent review

No agent calls are wired to Run All. Read `manual_agent_instructions.md` and give an evaluator the selected
case packet plus `report.md` when ready. Ask for the implicated facet, exact evidence references, hypothesis,
proposed improvement and the source check or experiment needed to verify it. Do not treat its diagnoses as labels.

The most useful hypotheses distinguish an inaccurate facet from an accurate but uninformative facet, a weighting
problem, a correct profile ignored by the selector, missing source evidence, and an unsettled rubric.


In [ ]:
display(Markdown((analysis.out / "manual_agent_instructions.md").read_text()))
print("Source-linked report:", analysis.out / "report.md")
print("Manifest with source hashes:", analysis.out / "manifest.json")
print("Review packets:", analysis.out / "cases")
